# V3.0 DF Experiment — EarlyStopping + MixUp + Reduced KL

**Changes from v2:**
- EarlyStopping (patience=15) — prevents overfitting after peak
- Epochs: 100 → 50 (saves 50% GPU time)
- KL annealing: λ_kl 0.1→0.02, cap at 0.5 (prevents evidence collapse)
- ORCU relaxation: λ_orcu 0.5→0.05, stages 5/30→3/10 (less aggressive)
- RandAugment(2,9) in data transforms (stronger augmentation)
- MixUp(α=0.2) for CE/Cumulative/SORD modes (combats memorization)
- Per-sample .npz export for all 5 modes

**Expected runtime**: ~1.5h on T4 x2 (vs v2's 2.5h).

## 1. Environment Setup

In [ ]:
!pip install transformers scikit-learn openpyxl scipy -q

# Clone updated code from GitHub
!rm -rf /kaggle/working/fluorosis_project
!git clone https://github.com/XiaoHG/fluorosis_project.git /kaggle/working/fluorosis_project

import os
assert os.path.isdir("/kaggle/working/fluorosis_project/06_Implementation"), \
    "Git clone failed!"
print("Code cloned successfully.")

In [ ]:
# Master Setup — cache pretrained weights while internet is available
import os, sys, json, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

print("Downloading pretrained weights...")
from transformers import ViTModel
_ = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
print("Weights cached.")

CODE_ROOT = "/kaggle/working/fluorosis_project/06_Implementation"
DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis-data-1"
OUTPUT_DIR = "/kaggle/working"
sys.path.insert(0, CODE_ROOT)

from data import create_dataloaders, DFDataset, get_transforms
from models import build_model
from losses import CombinedLoss
from eval import compute_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print("Setup complete.")

## 2. Data Verification

In [ ]:
def verify_df():
    ds_cls = DFDataset
    for split in ["train", "val", "test"]:
        ds = ds_cls(DATA_ROOT, split=split)
        labels = np.array([ds[i][1] for i in range(len(ds))])
        dist = {f"G{k}": int((labels == k).sum()) for k in range(4)}
        print(f"  DF {split:5s}: {len(ds):3d} samples | {dist}")

print("=== Data Verification ===")
verify_df()
print("OK.")

## 3. V3 Training Components

In [ ]:
# ---- MixUp (for CE/Cumulative/SORD modes) ----
def mixup_data(x, y, alpha=0.2):
    """MixUp augmentation. Returns (mixed_x, y_a, y_b, lam)."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """MixUp loss: lam * loss(pred, y_a) + (1-lam) * loss(pred, y_b)."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

print("MixUp helpers ready.")

In [ ]:
# ---- EarlyStopping ----
class EarlyStopping:
    """Stop training when validation metric stops improving."""
    def __init__(self, patience=15, min_delta=0.001, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = -float("inf") if mode == "max" else float("inf")
        self.counter = 0
        self.early_stop = False
        self.best_epoch = 0

    def __call__(self, metric, epoch):
        if self.mode == "max":
            improved = metric > self.best + self.min_delta
        else:
            improved = metric < self.best - self.min_delta
        if improved:
            self.best = metric
            self.counter = 0
            self.best_epoch = epoch
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

print("EarlyStopping ready.")

In [ ]:
# ---- V3 Training Function (EarlyStopping + MixUp + 50 epochs) ----
@torch.no_grad()
def evaluate_model(model, dataloader):
    model.eval()
    all_alpha, all_z, all_targets = [], [], []
    for images, targets in dataloader:
        images, targets = images.to(device), targets.to(device)
        alpha, z = model(images)
        if alpha is not None:
            all_alpha.append(alpha.cpu())
        all_z.append(z.cpu())
        all_targets.append(targets.cpu())
    alpha = torch.cat(all_alpha) if all_alpha else None
    z = torch.cat(all_z, dim=0)
    targets = torch.cat(all_targets, dim=0)
    return compute_metrics(alpha, z, targets)


def train_model_v3(task, data_root, output_dir, mode="edl_orcu",
                   batch_size=32, epochs=50,
                   lr_backbone=1e-4, lr_head=1e-3, weight_decay=0.05,
                   warmup_epochs=5,
                   kl_lambda=0.02, kl_anneal_cap=0.5,
                   lambda_orcu=0.05, lambda_reg=0.005, orcu_t=3.0,
                   stage_1_epochs=3, stage_2_epochs=10,
                   use_mixup=True, mixup_alpha=0.2,
                   patience=15, seed=42):
    """V3 training: EarlyStopping + MixUp + reduced epochs + relaxed KL/ORCU."""

    torch.manual_seed(seed)
    np.random.seed(seed)
    os.makedirs(output_dir, exist_ok=True)

    train_loader, val_loader, test_loader = create_dataloaders(
        data_root, task=task, batch_size=batch_size, num_workers=2)
    print(f"Data: train={len(train_loader.dataset)} val={len(val_loader.dataset)} "
          f"test={len(test_loader.dataset)}")

    model = build_model(task=task, mode=mode)
    model.to(device)
    model_type = type(model).__name__
    print(f"Model: {sum(p.numel() for p in model.parameters()):,} params | "
          f"Mode: {mode} | Type: {model_type}")

    mixup_modes = {"ce", "cumulative", "sord"}
    apply_mixup = use_mixup and mode in mixup_modes
    if apply_mixup:
        print(f"  MixUp: alpha={mixup_alpha} (active)")
    else:
        print(f"  MixUp: skipped (not applicable for {mode})")

    # ---- Criterion (v3 defaults) ----
    if mode == "ce":
        def criterion(a, z, t, e):
            loss = F.nll_loss(F.log_softmax(z, dim=-1), t)
            return loss, {"stage": 0, "L_ce": loss.item()}
    elif mode == "cumulative":
        from losses.cumulative_loss import CumulativeLoss
        cum_fn = CumulativeLoss(num_classes=4)
        def criterion(a, z, t, e):
            ol = model.ordinal_logits if hasattr(model, 'ordinal_logits') else z
            loss = cum_fn(ol, t)
            return loss, {"stage": 0, "L_cum": loss.item()}
    elif mode == "sord":
        from losses.orcu_loss import ORCULoss
        sord_fn = ORCULoss(num_classes=4, t=orcu_t, lambda_reg=0.0)
        def criterion(a, z, t, e):
            loss = sord_fn(z, t)
            return loss, {"stage": 0, "L_sord": loss.item()}
    elif mode == "edl":
        from losses.edl_loss import EDLLoss
        edl_fn = EDLLoss(num_classes=4, kl_lambda=kl_lambda, kl_anneal_cap=kl_anneal_cap)
        def criterion(a, z, t, e):
            loss = edl_fn(a, t, e, epochs)
            return loss, {"stage": 0, "L_edl": loss.item()}
    else:  # edl_orcu
        criterion = CombinedLoss(
            num_classes=4, lambda_orcu=lambda_orcu, lambda_kl=kl_lambda,
            kl_anneal_cap=kl_anneal_cap,
            orcu_t=orcu_t, orcu_lambda_reg=lambda_reg,
            stage_1_epochs=stage_1_epochs, stage_2_epochs=stage_2_epochs,
            total_epochs=epochs)
        print(f"  ORCU: stages={stage_1_epochs}/{stage_2_epochs}, "
              f"lambda_orcu={lambda_orcu}, lambda_reg={lambda_reg}")

    # ---- Optimizer & Scheduler ----
    bb_params, hd_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (bb_params if n.startswith("backbone") else hd_params).append(p)
    optimizer = optim.AdamW([
        {"params": bb_params, "lr": lr_backbone},
        {"params": hd_params, "lr": lr_head},
    ], weight_decay=weight_decay)

    warmup_sched = LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
    cosine_sched = CosineAnnealingLR(optimizer, T_max=epochs - warmup_epochs)
    scheduler = SequentialLR(optimizer, schedulers=[warmup_sched, cosine_sched],
                             milestones=[warmup_epochs])

    # ---- Training Loop ----
    early_stopper = EarlyStopping(patience=patience, mode="max")
    best_val_acc, best_state, best_epoch = 0.0, None, 0
    stopped_early = False
    history = []

    for epoch in range(epochs):
        model.train()
        epoch_losses = {"L_ce": 0.0, "L_cum": 0.0, "L_sord": 0.0,
                        "L_edl": 0.0, "L_orcu": 0.0, "L_total": 0.0}
        epoch_stage, n_batches = 0, 0

        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)

            if apply_mixup:
                images, targets_a, targets_b, lam = mixup_data(
                    images, targets, alpha=mixup_alpha)

            alpha_out, z = model(images)

            if apply_mixup:
                loss_a, loss_info = criterion(alpha_out, z, targets_a, epoch)
                loss_b, _ = criterion(alpha_out, z, targets_b, epoch)
                loss = lam * loss_a + (1 - lam) * loss_b
            else:
                loss, loss_info = criterion(alpha_out, z, targets, epoch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_stage = loss_info.get("stage", 0)
            for k in epoch_losses:
                if k in loss_info:
                    epoch_losses[k] += loss_info[k]
            n_batches += 1

        scheduler.step()
        for k in epoch_losses:
            epoch_losses[k] /= max(n_batches, 1)

        val_metrics = evaluate_model(model, val_loader)
        if val_metrics["acc"] > best_val_acc:
            best_val_acc = val_metrics["acc"]
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch

        history.append({
            "epoch": epoch, "stage": epoch_stage,
            "train_loss": epoch_losses,
            "val_metrics": {k: v for k, v in val_metrics.items()
                            if isinstance(v, (float, int)) and not k.startswith("evidence_class_")},
        })

        loss_key = next((k for k in ["L_total", "L_ce", "L_cum", "L_sord", "L_edl", "L_orcu"]
                         if epoch_losses.get(k, 0) != 0), "L_total")
        if (epoch + 1) % 10 == 0 or epoch == 0 or epoch == epochs - 1:
            print(f"[E{epoch+1:3d}/{epochs}] S={epoch_stage} | "
                  f"{loss_key}={epoch_losses.get(loss_key, 0):.4f} | "
                  f"Acc={val_metrics['acc']:.4f} F1={val_metrics['macro_f1']:.4f} "
                  f"QWK={val_metrics['qwk']:.4f} ECE={val_metrics['ece']:.4f} "
                  f"ES={early_stopper.counter}/{patience}")

        if early_stopper(val_metrics["acc"], epoch):
            print(f"  EarlyStopping at epoch {epoch+1} "
                  f"(best: epoch {best_epoch+1}, val_acc={best_val_acc:.4f})")
            stopped_early = True
            break

    if not stopped_early:
        print(f"  Completed {epochs} epochs (best: epoch {best_epoch+1}, "
              f"val_acc={best_val_acc:.4f})")

    # Load best and test
    model.load_state_dict(best_state)
    test_metrics = evaluate_model(model, test_loader)

    # Save
    torch.save({"model_state_dict": best_state, "test_metrics": test_metrics,
                "best_epoch": best_epoch, "stopped_early": stopped_early},
               os.path.join(output_dir, "best.pt"))
    with open(os.path.join(output_dir, "test_metrics.json"), "w") as f:
        json.dump({k: v for k, v in test_metrics.items() if isinstance(v, (float, int))},
                  f, indent=2)
    with open(os.path.join(output_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    print(f"\n[Test] Acc={test_metrics['acc']:.4f} F1={test_metrics['macro_f1']:.4f} "
          f"QWK={test_metrics['qwk']:.4f} ECE={test_metrics['ece']:.4f} "
          f"BestEp={best_epoch+1} EarlyStop={stopped_early}")
    return test_metrics, history

print("train_model_v3() ready.")

## 4. DF Main Experiment — 5 Modes (V3)

In [ ]:
print("=" * 60)
print("DF V3 Main Experiment — 5 Modes")
print("=" * 60)
print("V3: EarlyStopping(15) | MixUp(CE/Cum/SORD) | RandAugment")
print("    KL: λ=0.02 cap=0.5 | ORCU: λ=0.05 reg=0.005 stages=3/10")

df_results = {}
df_modes = ["ce", "cumulative", "sord", "edl", "edl_orcu"]

for mode in df_modes:
    print(f"\n{'='*60}")
    print(f"DF V3 | Mode: {mode}")
    print(f"{'='*60}")
    out_dir = os.path.join(OUTPUT_DIR, f"v3_df_{mode}")
    metrics, history = train_model_v3("df", DATA_ROOT, out_dir, mode=mode,
                                       epochs=50, batch_size=32, seed=42)
    df_results[mode] = {k: v for k, v in metrics.items() if isinstance(v, (float, int))}

    # Per-sample prediction export
    from eval import export_predictions, save_predictions
    train_loader, _, test_loader = create_dataloaders(
        DATA_ROOT, task="df", batch_size=32, num_workers=2)
    model = build_model(task="df", mode=mode)
    ckpt = torch.load(os.path.join(out_dir, "best.pt"),
                      map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    pred_data = export_predictions(model, test_loader, device)
    npz_path = os.path.join(out_dir, f"v3_df_{mode}_seed42_predictions.npz")
    save_predictions(pred_data, npz_path, mode=mode, task="df", seed=42)
    print(f"  -> Predictions: {npz_path}")

# Save
with open(os.path.join(OUTPUT_DIR, "v3_df_all_results.json"), "w") as f:
    json.dump(df_results, f, indent=2)

print("\n=== DF V3 Results ===")
hdr = f"{'Mode':<14} {'Acc':>8} {'F1':>8} {'QWK':>8} {'ECE':>8} {'SCE':>8} {'%Unim':>8}"
print(hdr)
print("-" * len(hdr))
for mode, m in df_results.items():
    print(f"{mode:<14} {m.get('acc',0):8.4f} {m.get('macro_f1',0):8.4f} "
          f"{m.get('qwk',0):8.4f} {m.get('ece',0):8.4f} {m.get('sce',0):8.4f} "
          f"{m.get('pct_unimodal',0):7.2%}")

## 5. 5-Fold Cross-Validation (V3)

In [ ]:
# ---- 5-Fold CV (V3 params) ----
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from scipy import stats

def run_cv_v3(data_root, methods=None, n_folds=5, epochs=50, seed=42):
    if methods is None:
        methods = ["ce", "cumulative", "edl", "edl_orcu"]

    label_ds = DFDataset(data_root, split="train", split_seed=seed)
    labels = np.array([label_ds[i][1] for i in range(len(label_ds))])

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    fold_results = {m: [] for m in methods}

    for fold_idx, (train_idx, val_idx) in enumerate(
            skf.split(np.zeros(len(label_ds)), labels)):
        print(f"\n{'='*50}")
        print(f"DF CV V3 Fold {fold_idx+1}/{n_folds}")
        print(f"{'='*50}")

        train_full = DFDataset(data_root, split="train", split_seed=seed)
        val_full = DFDataset(data_root, split="train", split_seed=seed)
        train_full.transform = get_transforms("df", is_train=True)
        val_full.transform = get_transforms("df", is_train=False)

        train_subset = Subset(train_full, train_idx)
        val_subset = Subset(val_full, val_idx)

        test_ds = DFDataset(data_root, split="test")
        test_ds.transform = get_transforms("df", is_train=False)

        train_loader = DataLoader(train_subset, batch_size=32, shuffle=True,
                                  num_workers=2, pin_memory=True)
        val_loader = DataLoader(val_subset, batch_size=32, shuffle=False,
                                num_workers=2, pin_memory=True)
        test_loader = DataLoader(test_ds, batch_size=32, shuffle=False,
                                 num_workers=2, pin_memory=True)

        for method in methods:
            print(f"  [{method}] training...")
            torch.manual_seed(seed + fold_idx * 100)
            np.random.seed(seed + fold_idx * 100)
            model = build_model(task="df", mode=method)
            model.to(device)

            if method == "ce":
                def crit(a, z, t, e):
                    loss = F.nll_loss(F.log_softmax(z, dim=-1), t)
                    return loss, {"stage": 0, "L_ce": loss.item()}
            elif method == "cumulative":
                from losses.cumulative_loss import CumulativeLoss
                cf = CumulativeLoss(num_classes=4)
                def crit(a, z, t, e):
                    ol = model.ordinal_logits if hasattr(model, 'ordinal_logits') else z
                    loss = cf(ol, t)
                    return loss, {"stage": 0, "L_cum": loss.item()}
            elif method == "edl":
                from losses.edl_loss import EDLLoss
                ef = EDLLoss(num_classes=4, kl_lambda=0.02, kl_anneal_cap=0.5)
                def crit(a, z, t, e):
                    loss = ef(a, t, e, epochs)
                    return loss, {"stage": 0, "L_edl": loss.item()}
            else:  # edl_orcu
                crit = CombinedLoss(
                    num_classes=4, lambda_orcu=0.05, lambda_kl=0.02,
                    kl_anneal_cap=0.5,
                    orcu_t=3.0, orcu_lambda_reg=0.005,
                    stage_1_epochs=3, stage_2_epochs=10, total_epochs=epochs)

            bb, hd = [], []
            for n, p in model.named_parameters():
                if not p.requires_grad:
                    continue
                (bb if n.startswith("backbone") else hd).append(p)
            optimizer = optim.AdamW(
                [{"params": bb, "lr": 1e-4}, {"params": hd, "lr": 1e-3}],
                weight_decay=0.05)
            ws = LinearLR(optimizer, start_factor=0.1, total_iters=5)
            cs = CosineAnnealingLR(optimizer, T_max=epochs - 5)
            scheduler = SequentialLR(optimizer, schedulers=[ws, cs], milestones=[5])

            es = EarlyStopping(patience=15, mode="max")
            best_val_acc, best_state = 0.0, None

            @torch.no_grad()
            def eval_cv(m, loader):
                m.eval()
                aa, zz, tt = [], [], []
                for im, tg in loader:
                    im, tg = im.to(device), tg.to(device)
                    a, z_ = m(im)
                    if a is not None:
                        aa.append(a.cpu())
                    zz.append(z_.cpu())
                    tt.append(tg.cpu())
                a = torch.cat(aa) if aa else None
                return compute_metrics(a, torch.cat(zz), torch.cat(tt))

            for epoch in range(epochs):
                model.train()
                for images, targets in train_loader:
                    images, targets = images.to(device), targets.to(device)
                    alpha, z = model(images)
                    loss, _ = crit(alpha, z, targets, epoch)
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                scheduler.step()

                vm = eval_cv(model, val_loader)
                if vm["acc"] > best_val_acc:
                    best_val_acc = vm["acc"]
                    best_state = copy.deepcopy(model.state_dict())
                if es(vm["acc"], epoch):
                    break

            model.load_state_dict(best_state)
            tm = eval_cv(model, test_loader)
            fold_results[method].append({
                "fold": fold_idx,
                **{k: v for k, v in tm.items() if isinstance(v, (float, int))}
            })
            print(f"    Acc={tm['acc']:.4f} F1={tm['macro_f1']:.4f} "
                  f"QWK={tm['qwk']:.4f} ECE={tm['ece']:.4f}")

    # Aggregate
    summary = {}
    metric_names = ["acc", "macro_f1", "qwk", "ece", "pct_unimodal", "u_ece", "auroc_u"]
    for method in methods:
        agg = {}
        for mn in metric_names:
            vals = [f[mn] for f in fold_results[method] if mn in f]
            agg[mn] = {"mean": float(np.mean(vals)), "std": float(np.std(vals)),
                       "values": vals}
        summary[method] = agg

    # Paired t-tests
    significance = {}
    for i, m1 in enumerate(methods):
        for j, m2 in enumerate(methods):
            if i >= j:
                continue
            pair_key = f"{m1}_vs_{m2}"
            significance[pair_key] = {}
            for mn in ["acc", "macro_f1", "qwk", "ece"]:
                v1 = [f[mn] for f in fold_results[m1]]
                v2 = [f[mn] for f in fold_results[m2]]
                t_stat, p_val = stats.ttest_rel(v1, v2)
                significance[pair_key][mn] = {
                    "t_statistic": float(t_stat), "p_value": float(p_val),
                    "significant": bool(p_val < 0.05)}

    return {"task": "df", "k": n_folds, "methods": methods,
            "fold_results": fold_results, "summary": summary,
            "significance": significance}

print("run_cv_v3() ready.")

# Run
print("\n" + "#" * 60)
print("# DF 5-Fold CV (V3)")
print("#" * 60)
cv_result = run_cv_v3(DATA_ROOT,
                      methods=["ce", "cumulative", "edl", "edl_orcu"],
                      n_folds=5, epochs=50)

with open(os.path.join(OUTPUT_DIR, "v3_cv_df.json"), "w") as f:
    json.dump(cv_result, f, indent=2)

print("\n=== V3 DF CV Summary ===")
print(f"{'Method':<14} {'Acc':>16} {'F1':>16} {'QWK':>16} {'ECE':>16}")
print("-" * 74)
for method, agg in cv_result["summary"].items():
    print(f"{method:<14} "
          f"{agg['acc']['mean']:.4f}+/-{agg['acc']['std']:.4f}  "
          f"{agg['macro_f1']['mean']:.4f}+/-{agg['macro_f1']['std']:.4f}  "
          f"{agg['qwk']['mean']:.4f}+/-{agg['qwk']['std']:.4f}  "
          f"{agg['ece']['mean']:.4f}+/-{agg['ece']['std']:.4f}")

print("\n  Significance (p < 0.05):")
for pair, tests in cv_result["significance"].items():
    sigs = [f"{m}(p={v['p_value']:.3f})" for m, v in tests.items()
            if v['significant']]
    print(f"    {pair}: {', '.join(sigs) if sigs else 'none'}")

## 6. Lambda Sweep — Extended Low Range (V3)

In [ ]:
# ---- Lambda Sweep: Extended low range ----
import itertools

print("EDL+ORCU Lambda Sweep — Extended Low Range")
print("  V2: λ_orcu ∈ {0.1, 0.3, 0.5} — all too high")
print("  V3: λ_orcu ∈ {0.01, 0.03, 0.05, 0.08, 0.10}")
print("       λ_reg  ∈ {0.005, 0.01}")
print(f"  epochs=50, runs={5*2}")

lambda_orcu_vals = [0.01, 0.03, 0.05, 0.08, 0.10]
lambda_reg_vals = [0.005, 0.01]
sweep_epochs = 50
sweep_seed = 42

train_loader, val_loader, test_loader = create_dataloaders(
    DATA_ROOT, task="df", batch_size=32, num_workers=2)

sweep_results = []
best_val_acc = 0.0
best_combo = None

@torch.no_grad()
def eval_sweep(m, loader):
    m.eval()
    aa, zz, tt = [], [], []
    for im, tg in loader:
        im, tg = im.to(device), tg.to(device)
        a, z = m(im)
        if a is not None:
            aa.append(a.cpu())
        zz.append(z.cpu())
        tt.append(tg.cpu())
    a = torch.cat(aa) if aa else None
    return compute_metrics(a, torch.cat(zz), torch.cat(tt))

for lam_orcu, lam_reg in itertools.product(lambda_orcu_vals, lambda_reg_vals):
    print(f"  λ_orcu={lam_orcu:.3f}, λ_reg={lam_reg:.4f} ...")
    torch.manual_seed(sweep_seed)
    np.random.seed(sweep_seed)
    model = build_model(task="df", mode="edl_orcu")
    model.to(device)

    criterion = CombinedLoss(
        num_classes=4, lambda_orcu=lam_orcu, lambda_kl=0.02,
        kl_anneal_cap=0.5,
        orcu_t=3.0, orcu_lambda_reg=lam_reg,
        stage_1_epochs=3, stage_2_epochs=10, total_epochs=sweep_epochs)

    bb, hd = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (bb if n.startswith("backbone") else hd).append(p)
    optimizer = optim.AdamW(
        [{"params": bb, "lr": 1e-4}, {"params": hd, "lr": 1e-3}],
        weight_decay=0.05)
    ws = LinearLR(optimizer, start_factor=0.1, total_iters=5)
    cs = CosineAnnealingLR(optimizer, T_max=sweep_epochs - 5)
    scheduler = SequentialLR(optimizer, schedulers=[ws, cs], milestones=[5])

    es = EarlyStopping(patience=15, mode="max")
    best_val, best_state = 0.0, None

    for epoch in range(sweep_epochs):
        model.train()
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            alpha, z = model(images)
            loss, _ = criterion(alpha, z, targets, epoch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()
        vm = eval_sweep(model, val_loader)
        if vm["acc"] > best_val:
            best_val = vm["acc"]
            best_state = copy.deepcopy(model.state_dict())
        if es(vm["acc"], epoch):
            break

    model.load_state_dict(best_state)
    tm = eval_sweep(model, test_loader)

    result = {
        "lambda_orcu": lam_orcu, "lambda_reg": lam_reg,
        "val_acc": best_val,
        "test_acc": tm["acc"], "test_qwk": tm["qwk"],
        "test_ece": tm["ece"], "test_unim": tm["pct_unimodal"],
        "test_u_ece": tm["u_ece"], "test_auroc_u": tm["auroc_u"],
    }
    sweep_results.append(result)

    if best_val > best_val_acc:
        best_val_acc = best_val
        best_combo = (lam_orcu, lam_reg)

    print(f"    Val={best_val:.4f} Test={tm['acc']:.4f} QWK={tm['qwk']:.4f} "
          f"ECE={tm['ece']:.4f} Unim={tm['pct_unimodal']:.2%} "
          f"AUROC(u)={tm['auroc_u']:.4f}")

# Summary
print(f"\n{'='*80}")
print("Lambda Sweep V3 Results (sorted by Val Acc)")
print(f"{'='*80}")
hdr = (f"{'λ_orcu':>8} {'λ_reg':>8} {'Val':>8} {'Test':>8} "
       f"{'QWK':>8} {'ECE':>8} {'Unim':>8} {'AUROC(u)':>8}")
print(hdr)
print("-" * 72)
for r in sorted(sweep_results, key=lambda x: x["val_acc"], reverse=True):
    print(f"{r['lambda_orcu']:8.3f} {r['lambda_reg']:8.4f} "
          f"{r['val_acc']:8.4f} {r['test_acc']:8.4f} {r['test_qwk']:8.4f} "
          f"{r['test_ece']:8.4f} {r['test_unim']:7.2%} {r['test_auroc_u']:8.4f}")

print(f"\nBest: λ_orcu={best_combo[0]}, λ_reg={best_combo[1]} "
      f"(Val Acc={best_val_acc:.4f})")

with open(os.path.join(OUTPUT_DIR, "v3_lambda_sweep.json"), "w") as f:
    json.dump({"results": sweep_results, "best_combo": list(best_combo)}, f, indent=2)
print("Saved to v3_lambda_sweep.json")

## 7. Results Summary

In [ ]:
print("\n" + "=" * 78)
print("V3 FINAL RESULTS SUMMARY")
print("=" * 78)

all_summary = {}

for fname, key in [("v3_df_all_results.json", "df_all_results"),
                   ("v3_cv_df.json", "cv_df"),
                   ("v3_lambda_sweep.json", "lambda_sweep")]:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        with open(fpath) as f:
            data = json.load(f)
            if key == "cv_df":
                all_summary[key] = data.get("summary", {})
                all_summary["cv_significance"] = data.get("significance", {})
            else:
                all_summary[key] = data

# Main table
print("\n--- DF Main Experiment (V3) ---")
if "df_all_results" in all_summary:
    results = all_summary["df_all_results"]
    hdr = f"{'Mode':<14} {'Acc':>8} {'F1':>8} {'QWK':>8} {'ECE':>8} {'SCE':>8} {'%Unim':>8}"
    print(hdr)
    print("-" * len(hdr))
    for mode, m in results.items():
        print(f"{mode:<14} {m.get('acc',0):8.4f} {m.get('macro_f1',0):8.4f} "
              f"{m.get('qwk',0):8.4f} {m.get('ece',0):8.4f} "
              f"{m.get('sce',0):8.4f} {m.get('pct_unimodal',0):7.2%}")

# CV
print("\n--- 5-Fold CV (V3) ---")
if "cv_df" in all_summary:
    for method, agg in all_summary["cv_df"].items():
        if isinstance(agg, dict) and "acc" in agg:
            print(f"  {method:<14}: Acc={agg['acc']['mean']:.4f}+/-{agg['acc']['std']:.4f}  "
                  f"QWK={agg['qwk']['mean']:.4f}+/-{agg['qwk']['std']:.4f}")

# Lambda top-3
print("\n--- Lambda Sweep Top-3 (V3) ---")
if "lambda_sweep" in all_summary:
    top3 = sorted(all_summary["lambda_sweep"].get("results", []),
                  key=lambda x: x.get("val_acc", 0), reverse=True)[:3]
    for r in top3:
        print(f"  λ_orcu={r['lambda_orcu']:.3f} λ_reg={r['lambda_reg']:.4f}: "
              f"Val={r['val_acc']:.4f} Test={r['test_acc']:.4f} QWK={r['test_qwk']:.4f}")

# Save master
with open(os.path.join(OUTPUT_DIR, "v3_master_summary.json"), "w") as f:
    json.dump(all_summary, f, indent=2)

print(f"\nAll results saved to {OUTPUT_DIR}/v3_master_summary.json")
print("\n=== V3 EXPERIMENTS COMPLETE ===")
print("Download /kaggle/working/ for all checkpoints and result files.")